In [16]:
# Instalação das bibliotecas necessárias
!pip install scikit-learn pandas google-generativeai
!pip install -U google-generativeai

In [13]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import google.generativeai as genai



In [9]:
# =====================================================================
# PARTE 1: Geração de Dados Realistas (ETL Simulado)
# =====================================================================
np.random.seed(42) # Mantém os valores consistentes a cada execução

n_amostras = 1000

# 1. Gerando dados de AGVs Saudáveis (70% da frota)
# Temperaturas em torno de 80°C, baterias variadas, vibração baixa (~2.5Hz)
n_saudaveis = int(n_amostras * 0.7)
temp_saudavel = np.random.normal(loc=80, scale=5, size=n_saudaveis)
bat_saudavel = np.random.uniform(low=40, high=100, size=n_saudaveis)
vib_saudavel = np.random.normal(loc=2.5, scale=0.5, size=n_saudaveis)
status_saudavel = np.zeros(n_saudaveis) # 0 = Saudável

# 2. Gerando dados de AGVs com Falha (30% da frota)
# Temperaturas altas (~110°C), baterias mais baixas, vibração alta (~6.5Hz)
n_falhas = n_amostras - n_saudaveis
temp_falha = np.random.normal(loc=110, scale=8, size=n_falhas)
bat_falha = np.random.uniform(low=5, high=50, size=n_falhas)
vib_falha = np.random.normal(loc=6.5, scale=1.2, size=n_falhas)
status_falha = np.ones(n_falhas) # 1 = Falha Crítica

# 3. Consolidando no DataFrame e embaralhando
df_agv = pd.DataFrame({
    'Temperatura_Motor': np.concatenate([temp_saudavel, temp_falha]),
    'Nivel_Bateria': np.concatenate([bat_saudavel, bat_falha]),
    'Vibracao_Chassi': np.concatenate([vib_saudavel, vib_falha]),
    'Status_Falha': np.concatenate([status_saudavel, status_falha])
})
df_agv = df_agv.sample(frac=1, random_state=42).reset_index(drop=True)

print("--- Amostra do Dataset Gerado ---")
print(df_agv.head(), "\n")



--- Amostra do Dataset Gerado ---
   Temperatura_Motor  Nivel_Bateria  Vibracao_Chassi  Status_Falha
0          82.716801      97.108707         2.775743           0.0
1         114.104683      15.175640         5.388343           1.0
2         104.868147      15.368750         5.999160           1.0
3          77.131690      80.052994         1.903037           0.0
4          74.376790      83.895371         3.069939           0.0 



In [10]:
# =====================================================================
# PARTE 2: Treinamento da Árvore de Decisão
# =====================================================================
X = df_agv[['Temperatura_Motor', 'Nivel_Bateria', 'Vibracao_Chassi']]
y = df_agv['Status_Falha']

# Separando em dados de treino (80%) e teste (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_agv = DecisionTreeClassifier(max_depth=4, random_state=42)
modelo_agv.fit(X_train, y_train)

# Avaliando o modelo
acuracia = accuracy_score(y_test, modelo_agv.predict(X_test))
print(f"--- Performance do Modelo ---")
print(f"Acurácia da Árvore de Decisão: {acuracia * 100:.2f}%\n")

# Simulando um AGV com problemas para passar para o Gemini
nova_leitura = pd.DataFrame({
    'Temperatura_Motor': [108.5],
    'Nivel_Bateria': [22.0],
    'Vibracao_Chassi': [5.8]
})

previsao = modelo_agv.predict(nova_leitura)[0]
diagnostico_tecnico = "Alerta: Falha Crítica Iminente" if previsao == 1 else "Operação Saudável"

print(f"--- Diagnóstico Operacional ---")
print(f"Leitura -> Temp: {nova_leitura['Temperatura_Motor'][0]}°C | Bat: {nova_leitura['Nivel_Bateria'][0]}% | Vib: {nova_leitura['Vibracao_Chassi'][0]}Hz")
print(f"Decisão: {diagnostico_tecnico}\n")



--- Performance do Modelo ---
Acurácia da Árvore de Decisão: 99.50%

--- Diagnóstico Operacional ---
Leitura -> Temp: 108.5°C | Bat: 22.0% | Vib: 5.8Hz
Decisão: Alerta: Falha Crítica Iminente



In [19]:
# =====================================================================
# PARTE 3: Integração com IA Generativa (Gemini)
# =====================================================================
from google.colab import userdata
import google.generativeai as genai

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

model = genai.GenerativeModel('gemini-2.5-flash')

prompt = f"""
Você é a inteligência co-piloto do projeto SmartLog. Nosso sistema especialista (Árvore de Decisão)
analisou a telemetria de um AGV logístico em rota e gerou o seguinte diagnóstico:

[DADOS DA TELEMETRIA]
- Temperatura do Motor: {nova_leitura['Temperatura_Motor'][0]:.1f}°C
- Nível de Bateria Atual: {nova_leitura['Nivel_Bateria'][0]:.1f}%
- Frequência de Vibração do Chassi: {nova_leitura['Vibracao_Chassi'][0]:.1f}Hz

[DIAGNÓSTICO DO SISTEMA]
- Status: {diagnostico_tecnico}

Gere um relatório operacional conciso (máximo de 2 parágrafos e alguns bullet points) contendo:
1. Uma explicação técnica de como esses três sensores indicam o estado mecânico/elétrico do AGV.
2. Duas ações corretivas imediatas para a central de controle.
"""

resposta_interpretativa = model.generate_content(prompt)

print(f"--- Relatório Interpretativo (Gemini Co-Piloto) ---")
print(resposta_interpretativa.text)

--- Relatório Interpretativo (Gemini Co-Piloto) ---
**Relatório Operacional SmartLog - Alerta de AGV**

O diagnóstico de "Alerta: Falha Crítica Iminente" é justificado pela combinação de dados telemétricos que indicam uma degradação acelerada no estado mecânico e elétrico do AGV. A **Temperatura do Motor de 108.5°C** é alarmantemente alta, superando limites operacionais seguros e sugerindo sobrecarga severa, atrito excessivo (e.g., rolamentos desgastados) ou falha no sistema de refrigeração. Motores operando a essas temperaturas correm risco iminente de danos permanentes às bobinas e componentes internos, levando a uma falha total. Simultaneamente, a **Frequência de Vibração do Chassi de 5.8Hz** corrobora um problema mecânico subjacente, possivelmente um desbalanceamento, componentes soltos ou desgaste em rolamentos ou eixos, o que pode estar contribuindo diretamente para o esforço excessivo do motor e seu consequente superaquecimento.

Adicionalmente, o **Nível de Bateria Atual de 22.